In [6]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
non_music_path = "kaggle/input/dataset/non_music"
music_path = "dkaggle/input/dataset/music"
test_music_path = "dataset2/test/music"
test_non_music_path = "dataset2/test/non_music"

In [8]:
non_music_files = [os.path.join(non_music_path, f) for f in os.listdir(non_music_path)]
music_files = [os.path.join(music_path, f) for f in os.listdir(music_path)]
test_music_files = [os.path.join(test_music_path, f) for f in os.listdir(test_music_path)]
test_non_music_files = [os.path.join(test_non_music_path, f) for f in os.listdir(test_non_music_path)]


In [9]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path)
    spectrogram = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)  # Compute mel spectrogram
    spectrogram = librosa.power_to_db(spectrogram, ref=np.max)  # Convert to decibels
    return spectrogram

In [10]:
non_music_features = np.array([extract_features(f) for f in non_music_files])

In [11]:
music_features = np.array([extract_features(f) for f in music_files])

In [12]:
test_music_features = np.array([extract_features(f) for f in test_music_files])

In [13]:
test_non_music_features = np.array([extract_features(f) for f in test_non_music_files])

In [14]:
non_music_labels = np.zeros(len(non_music_files))
music_labels = np.ones(len(music_files))
test_non_music_labels = np.zeros(len(test_non_music_files))
test_music_labels = np.ones(len(test_music_files))

In [15]:
X_train = np.concatenate((non_music_features, music_features))
y_train = np.concatenate((non_music_labels, music_labels))
X_test = np.concatenate((test_non_music_features, test_music_features))
y_test = np.concatenate((test_non_music_labels, test_music_labels))

In [16]:
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

In [34]:
model = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=X_train.shape[1:]),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])


In [35]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [36]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())


[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 4504990065811101723
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 2908120679
locality {
  bus_id: 1
  links {
  }
}
incarnation: 1306406692121502260
physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1050, pci bus id: 0000:01:00.0, compute capability: 6.1"
xla_global_id: 416903419
]


In [37]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


Physical devices cannot be modified after being initialized


In [38]:
tf.config.set_visible_devices(gpus[0], 'GPU')  # Select the first GPU


In [39]:
tf.keras.backend.clear_session()


In [40]:
model.fit(X_train, y_train, epochs=10, batch_size=8, validation_split=0.2)

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [41]:
y_pred = (model.predict(X_test) > 0.9).astype("int32")
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

313/313 [==============================] - 161s 18ms/step
Accuracy: 0.4964


In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
#model.save("CNNmodelForMp3_98%.h5")